# From Strong Form to Weak Form

This notebook demonstrates the fundamental transformation from a strong form PDE into a weak form, which is the mathematical foundation of the Finite Element Method. We will convert a 1D bar equilibrium equation into its weak form using the method of weighted residuals and integration by parts.

In [1]:
import sympy as sp
from IPython.display import Math, display
from symbolic_fem_workbench import (
    make_field_1d, weighted_residual, integrate_divergence_1d,
    drop_dirichlet_boundary, apply_neumann_flux, split_linear_weak_form,
)

## Step 1: Define the problem

We consider a 1D bar under axial loading. The strong form of the equilibrium equation is:
$$\frac{d}{dx}\left(EA\frac{du}{dx}\right) + q = 0$$

where:
- $u(x)$ is the axial displacement
- $E$ is Young's modulus
- $A$ is the cross-sectional area
- $q$ is a distributed load
- Domain: $x \in [0, L]$

In [2]:
x, L = sp.symbols("x L", positive=True)
E, A = sp.symbols("E A", positive=True)
q, P = sp.symbols("q P", real=True)

u = make_field_1d("u", x)
v = make_field_1d("v", x)
domain = (x, 0, L)

print("Trial field u(x):", u)
print("Test field v(x):", v)
print("Domain:", domain)

Trial field u(x): u(x)
Test field v(x): v(x)
Domain: (x, 0, L)


## Step 2: Build the weighted residual

We multiply the strong form by a test function $v(x)$ (also called a weight function) and integrate over the domain. This is the method of weighted residuals:
$$\int_0^L \left[\frac{d}{dx}\left(EA\frac{du}{dx}\right) + q\right] v \, dx = 0$$

In [3]:
flux = E * A * sp.diff(u, x)
strong_form_lhs = sp.diff(flux, x) + q

display(Math(r"\text{Strong form: } " + sp.latex(strong_form_lhs) + " = 0"))

wr = weighted_residual(strong_form_lhs, 0, v, domain)
display(Math(r"\text{Weighted residual: } " + sp.latex(wr.as_expression()) + " = 0"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Step 3: Integration by parts

We now apply integration by parts to shift one derivative from $u$ to $v$. This lowers the regularity requirement on $u$ and introduces natural boundary conditions:
$$\int_0^L \frac{d}{dx}\left(EA\frac{du}{dx}\right) v \, dx = \left[EA\frac{du}{dx}v\right]_0^L - \int_0^L EA\frac{du}{dx}\frac{dv}{dx} \, dx$$

In [4]:
domain_term, (left_bc, right_bc) = integrate_divergence_1d(flux, v, x, domain)

print("Domain integral (after IBP):")
display(Math(sp.latex(domain_term.as_integral())))

print("\nLeft boundary term:")
display(Math(sp.latex(left_bc.evaluate())))

print("\nRight boundary term:")
display(Math(sp.latex(right_bc.evaluate())))

Domain integral (after IBP):


<IPython.core.display.Math object>


Left boundary term:


<IPython.core.display.Math object>


Right boundary term:


<IPython.core.display.Math object>

## Step 4: Apply boundary conditions

**Dirichlet BC (Essential):** The test function must vanish where displacement is prescribed.
At $x=0$: $u(0) = 0$ (homogeneous Dirichlet) $\Rightarrow$ $v(0) = 0$

**Neumann BC (Natural):** The prescribed flux/traction appears as a source term.
At $x=L$: Prescribed force $P$ $\Rightarrow$ $EA\frac{du}{dx}\big|_x=L = P$

In [5]:
# Homogeneous Dirichlet at x=0: test function vanishes
left_val = drop_dirichlet_boundary(left_bc, v)
print("Left BC (Dirichlet, v=0 at x=0):", left_val)

# Neumann at x=L: prescribed traction P
right_val = apply_neumann_flux(right_bc, flux_expr=flux, prescribed_flux=P)
print("Right BC (Neumann, flux=P at x=L):", right_val)

Left BC (Dirichlet, v=0 at x=0): 0
Right BC (Neumann, flux=P at x=L): -P*v(L)


## Step 5: Assemble and split the weak form

We now assemble all terms into a single weak form and decompose it into bilinear and linear parts:
$$a(u, v) + F(v) = 0$$
where $a(u,v)$ is bilinear in $u$ and $v$, and $F(v)$ is linear in $v$.

In [6]:
source_integral = sp.Integral(q * v, domain)
weak_expr = domain_term.as_integral() - source_integral + left_val + right_val

display(Math(r"\text{Full weak form expression: } " + sp.latex(weak_expr) + " = 0"))

weak_form = split_linear_weak_form(weak_expr, u, v)
print("\nBilinear form a(u,v):")
display(Math(sp.latex(weak_form.bilinear)))
print("\nLinear form F(v):")
display(Math(sp.latex(weak_form.linear)))

<IPython.core.display.Math object>


Bilinear form a(u,v):


<IPython.core.display.Math object>


Linear form F(v):


<IPython.core.display.Math object>

## Exercise

**Question:** Modify the problem to have Dirichlet conditions on both ends ($u(0) = 0$, $u(L) = 0$) with a distributed source term $q$. How does the boundary condition handling change? What happens to the natural boundary term contributions?

In [7]:
# Exercise: Modify the problem to have Dirichlet conditions on both ends:
# u(0) = 0, u(L) = 0, with source term q.
# What changes in the boundary condition handling?

